In [ ]:
%reset -f

import numpy as np
import scipy.optimize as SciOpt
from numpy import random as rnd
from numpy import linalg as LA
import matplotlib.pyplot as plt
import matplotlib
import time

matplotlib.rcParams.update({'font.size': 18})

########################################################################
### Using Maximum Likelihood function as a parameter Estimator (MLE) ###
########################################################################

In [ ]:
### generate data for an exponential decay "experiment", ###
### f(t) = exp(-t/tau)/tau                               ### 

NSAMPLES = 10000
TAU      = 2.2
t        = rnd.exponential(TAU,NSAMPLES); 

n,b,p = plt.hist(t)
plt.show()

In [ ]:
# define some functions

# returns probability density for RV and parameter

my_pdf = lambda data, tau : np.exp(-data/tau)/tau

# returns log-likelihood term for RV and parameter

logL = lambda data, tau : np.log(my_pdf(data,tau))

# The cost function is just the sum of all log-likelihood
# terms, "-" sign because optimizer routines seek minimums

costfun = lambda tau: -sum(logL(t,tau)) 

In [ ]:
# Do the fit using a generic simplex algorithm
# Note: output is an array; may minimize wrt > 1 parameter

result  = SciOpt.minimize(costfun,2.1,method='Nelder-Mead')
print("The estimated parameter is: %8.5f" % float(result.x[0])) 

In [ ]:
# Compare the analytical result for MLE

mle_analytical = np.mean(t)
std_analytical = np.std(t)/np.sqrt(NSAMPLES)   # 'error in the mean'

print("The estimate is : %8.5f +- %8.5f " % (mle_analytical,std_analytical))

In [ ]:
## now, repeat this experiment a bunch of times ###

NEXP = 500            # number of experiments
accu = np.zeros(NEXP) # data accumulator

for ii in np.linspace(1,NEXP,NEXP,dtype=int):

    # Generate NSAMPLES exponential RVs
    t = rnd.exponential(TAU, NSAMPLES)
    
    # Estimate decay constant using MLE
    result = SciOpt.minimize(costfun, 2.1, method='Nelder-Mead')
    
    accu[ii-1] = result.x[0]
    
print('Estimator standard deviation is: %10.5f' % float(np.std(accu)))

In [ ]:
# Plot the results

fig = plt.figure(figsize=(10,5))

n,b,p = plt.hist(accu,100,range=(1.5,2.5))

tminus = np.mean(accu)-np.std(accu)
tplus  = np.mean(accu)+np.std(accu)

plt.plot([tminus,tminus],[0.0,500.0],color='r')
plt.plot([tplus, tplus], [0.0,500.0],color='r')
plt.axis([1.5,2.5,0.0,120.0])
plt.xlabel("MLE Estimates for $\\tau$")
plt.ylabel("Counts at this value")
plt.show()